# 12 · Leave-One-Dataset-Out — NHANES III Held Out (3 seeds)

Trains the full method on OAI, MRKR, and Mendeley, and evaluates on the held-out NHANES III set across three random seeds.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import pandas as pd, numpy as np
from pathlib import Path
import torch
if 'training_lib' in sys.modules: importlib.reload(sys.modules['training_lib'])
import training_lib as T
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> Change runtime type -> GPU.')
print('GPU:', torch.cuda.get_device_name(0))
manifest = T.prepare_local_data()
print(f'Manifest: {len(manifest):,} rows')

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB
Copied array in 36s
Loaded array (61558, 224, 224) in 1s
Manifest: 61,558 rows


In [2]:
def make_splits(manifest, test_dataset, seed=0):
    pool=manifest[manifest['dataset']!=test_dataset].copy()
    test=manifest[manifest['dataset']==test_dataset].copy()
    if 'split' in pool.columns and pool['split'].isin(['train','val']).any():
        tr=pool[pool['split'].isin(['train','TRAIN'])]; va=pool[pool['split'].isin(['val','VAL','validation'])]
        if len(va)==0: va=pool.sample(frac=0.15,random_state=seed); tr=pool.drop(va.index)
    else:
        va=pool.sample(frac=0.15,random_state=seed); tr=pool.drop(va.index)
    return tr.reset_index(drop=True), va.reset_index(drop=True), test.reset_index(drop=True)
print('Split helper ready.')

Split helper ready.


## Execution

In [3]:
fk='fold3_test_nhanes3'; test_ds=config.LODO_FOLDS[fk]; mech=dict(config.FULL_METHOD)
try:
    _sw=pd.read_csv(str(config.RESULTS_DIR/'grl_lambda_sweep.csv'))
    _best=float(_sw.loc[_sw['external_acc5'].idxmax(),'grl_lambda_max'])
    mech['grl_lambda_max']=_best; print(f'Using swept GRL lambda: {_best}')
except Exception: print('No sweep file; using config GRL_LAMBDA_MAX')
results=[]
for seed in config.SEEDS:
    run=f'{fk}_seed{seed}'
    tr,va,te=make_splits(manifest,test_ds,seed=seed)
    print(f'--- {run} (train={len(tr):,} val={len(va):,} test={len(te):,}) ---',flush=True)
    r=T.run_training(run,tr,va,te,mech,seed,config.CKPT_DIR,config.RESULTS_DIR,log_fn=print)
    results.append(r)
df=pd.DataFrame(results)
print(f'External acc5: {df["external_acc5"].mean():.4f} +/- {df["external_acc5"].std():.4f}')
print(f'External QWK : {df["external_qwk"].mean():.4f}')
print(f'Gap          : {df["gap"].mean():.4f}')
df.to_csv(str(config.RESULTS_DIR/f'{fk}_results.csv'),index=False)

Using swept GRL lambda: 0.5
--- fold3_test_nhanes3_seed0 (train=48,257 val=8,516 test=4,785) ---
Downloading: "https://download.pytorch.org/models/convnext_large-ea097f82.pth" to /root/.cache/torch/hub/checkpoints/convnext_large-ea097f82.pth


100%|██████████| 755M/755M [00:03<00:00, 223MB/s]


  [fold3_test_nhanes3_seed0] ep1/18 loss=3.260 tr=0.402 val=0.404 gap=-0.002 qwk=0.259 grl=0.00 (130s)
  [fold3_test_nhanes3_seed0] ep2/18 loss=2.791 tr=0.456 val=0.434 gap=+0.021 qwk=0.390 grl=0.14 (114s)
  [fold3_test_nhanes3_seed0] ep3/18 loss=2.587 tr=0.487 val=0.461 gap=+0.026 qwk=0.477 grl=0.25 (113s)
  [fold3_test_nhanes3_seed0] ep4/18 loss=2.473 tr=0.504 val=0.483 gap=+0.021 qwk=0.538 grl=0.34 (113s)
  [fold3_test_nhanes3_seed0] ep5/18 loss=2.382 tr=0.525 val=0.492 gap=+0.032 qwk=0.538 grl=0.40 (113s)
  [fold3_test_nhanes3_seed0] ep6/18 loss=2.289 tr=0.525 val=0.495 gap=+0.030 qwk=0.571 grl=0.44 (113s)
  [fold3_test_nhanes3_seed0] ep7/18 loss=2.175 tr=0.534 val=0.502 gap=+0.032 qwk=0.576 grl=0.47 (114s)
  [fold3_test_nhanes3_seed0] ep8/18 loss=2.116 tr=0.532 val=0.511 gap=+0.021 qwk=0.573 grl=0.48 (112s)
  [fold3_test_nhanes3_seed0] ep9/18 loss=2.022 tr=0.537 val=0.518 gap=+0.019 qwk=0.574 grl=0.49 (113s)
  [fold3_test_nhanes3_seed0] ep10/18 loss=1.965 tr=0.539 val=0.522 gap=+0